# 0. Pyroton V2 Single EOT Run B

Attach the private `tunastuna/pyroton-v2-single-eot-inputs` dataset. Uses one visible T4, the 89 sealed pairs, and two ORPO epochs. Uses completion-only chosen NLL. The odds-ratio term already uses completion labels.

In [ ]:
# 1. Validate the sealed training inputs

from hashlib import sha256
from pathlib import Path
import subprocess
import sys

SEALED_SHA256 = {
    "kaggle_step3.py": "3d76b68f79e7dcdc569de68ecf5c328d77c939ba95a57abd0d00bea3a6313e07",
    "orpo-length-matched.jsonl": "16908db487a509bdf8c3c7401843dc0b97c95eae9f6820f3cbe8c80c3bb18cc9",
    "orpo_completion_only.py": "0d002da1037fd2cf2ceb8d893cadb034d6760cb6c56ab435a9c310ce4a70a270",
    "quality-report.json": "a576929001581a8cc6c1fe5101e55bb69911f5c4d7c4c7db47d62962a20da595",
    "stock-noise.json": "fd64b7c4cf653eed90782d091f0463b4a970c16588fd790b443094134563c58c",
    "train_step3_pilot.py": "b26d6777f6b9262224db54664b115b0a3f33f87633f88c5493a94f49d4c66c2c"
}

def find_sealed_inputs(root):
    mounts = sorted(path for path in root.rglob("pyroton-v2-single-eot-inputs") if path.is_dir())
    if len(mounts) != 1:
        raise RuntimeError(f"Expected one pyroton-v2-single-eot-inputs mount under {root}, found {len(mounts)}.")
    mount = mounts[0]
    for name, expected in SEALED_SHA256.items():
        path = mount / name
        if not path.is_file():
            raise FileNotFoundError(f"Missing sealed input: {name}")
        actual = sha256(path.read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError(f"SHA-256 mismatch for sealed input: {name}")
    return mount


inputs = find_sealed_inputs(Path("/kaggle/input"))
command = [sys.executable, str(inputs / "kaggle_step3.py")]
options = ["--inputs", str(inputs), "--working", "/kaggle/working", *['--single-eot', '--completion-only-nll']]
subprocess.run([*command, "validate", *options], check=True)


In [ ]:
# 2. Train two ORPO epochs

subprocess.run([*command, "train", *options], check=True)


In [ ]:
# 3. Export epoch adapters and first-step metrics

import json
subprocess.run([*command, "export", *options], check=True)
working = Path("/kaggle/working/single-eot-completion-only")
metrics = json.loads((working / "training-metrics.json").read_text())
first_step = next(row for row in metrics["loss_and_margin_log"] if row.get("step") == 1 and "loss" in row)
(working / "first-step.json").write_text(json.dumps(first_step, indent=2) + "\n")
